# 04 — The checker: deciding each highlighted piece in context

**Question:** if a language model reads each piece the highlighter flagged, together with its neighbouring text and the
label's definition from the taxonomy, does it keep the true labels and drop the wrong flags — on **every** list,
including lists no model was trained on?

```
record → lookup (codes) → highlighter (units + trigger words, notebook 03) → CHECKER (this notebook) → final labels
```

| Step | What | Status |
|---|---|---|
| 4a / 4b | Whole-record models (notebooks 01, 02) — best on new lists: ML-part F1 0.55, precision 0.46 | done |
| B | Highlighter (notebook 03) — flags 94.8% of the text layer's labels, 48% of flags right | done |
| **C** | **Checker (this notebook)** | **now** |

**The checker** (`scripts/checker.py`): `Qwen/Qwen3-4B-Instruct-2507` (open licence, Apache-2.0), loaded in 4-bit on the
laptop's RTX 4070 — **no data leaves the machine**. For every flagged (unit, label) pair it gets one question:

* the label's **definition, coverage and boundary** (taxonomy column C, English);
* the **field** the unit comes from (Remarks, Occupations, …);
* the **unit** itself, plus the unit before and after it as context;
* the question: does this text support assigning the label to the listed person or entity?

It answers yes or no. We read the **probability of "yes"** from the model's first answer word (one model pass, no text
generation), so every answer is a score between 0 and 1. A record gets a label when **any** of its flagged units scores
≥ 0.5.

**Nothing is trained on our labels.** The model never saw these records or lists, so there is no seen / unseen split:
every list is "new" to it. The 0.5 cut-off is fixed in advance. (A second view picks a cut-off per label from the
*other* unseen-list folds, to show what tuning could add.)

**Scope:** the same 15 labels as notebook 03. The 5 labels not covered yet and the 3 lookup-only labels are excluded.

**Caveat:** the reference labels were made by two LLM labellers (agreed + decided disputes), so a "wrong" answer may
sometimes be a reference mistake. The human answer sheet from notebook 03 is the independent check.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120, "display.width", 220, "display.precision", 3)

RESEARCH = Path.cwd().parent
sys.path.insert(0, str(RESEARCH / "scripts"))
from experimentKit import Data, crossfit_predict, label_tiers, ml_part, prf, wilson
from highlighter import EXTRA_TRIGGERS, TAXONOMY_TRIGGERS, compile_triggers, flag_units, split_units
from checker import MODEL_ID, PROMPT_VERSION, Checker, label_definitions, questions, role_questions

CACHE = RESEARCH / "cache"
d = Data()
df, Y, L = d.df, d.Y, d.L
TIERS = label_tiers(d)
SCOPE = [i for i in range(len(d.labels)) if TIERS[i] != "lookup only" and i not in d.rare]
NEED = Y & (1 - L)
in_scope = lambda lexicon: {k: v for k, v in lexicon.items() if d.labels.index(k) in SCOPE}
TRIGGERS = compile_triggers(in_scope(TAXONOMY_TRIGGERS), in_scope(EXTRA_TRIGGERS))
DEFS = label_definitions()

UNITS = [split_units(t) for t in df["text"]]
FLAGS = [flag_units(u, TRIGGERS) for u in UNITS]
ROWS = pd.DataFrame([{"record": r, "unit_index": k, "label": label, "field": UNITS[r][k][0], "unit": UNITS[r][k][1],
                      "trigger words": ", ".join(hits)}
                     for r, fl in enumerate(FLAGS) for k, label, hits in fl])
ROWS["prompt"] = [p for r, fl in enumerate(FLAGS) for p in questions(UNITS[r], fl, DEFS)]
print(f"{len(ROWS):,} questions for {ROWS['record'].nunique()} records; model: {MODEL_ID}")
print("\nexample question:\n" + "-" * 100 + "\n" + ROWS.loc[ROWS["label"] == "Crime-Corruption", "prompt"].iloc[0] + "\n" + "-" * 100)

3,430 questions for 642 records; model: Qwen/Qwen3-4B-Instruct-2507

example question:
----------------------------------------------------------------------------------------------------
Risk label: Crime-Corruption
Definition: Reporting of conduct or a specific allegation involving abuse of public power or entrusted authority for an improper personal or organizational benefit.
Coverage: Giving/receiving bribes, graft, kickbacks or improper payments, abuse of office, influence peddling and diversion of contracts/resources for private benefit. Embezzlement/misappropriation may also qualify when entrusted-authority abuse is reported. Mentioning an official or PEP alone is insufficient.
Boundary and overlap: A specific attributed allegation or supported conduct description is sufficient for reporting classification, not independent proof. Political criticism, inefficiency, an unexplained gift or rhetorical use of “corrupt” is insufficient. Reported appropriation may also receive Crime-Th

## 1. Run the checker
Answers are cached in `cache/checker_pyes.csv`, so re-running the notebook does not re-ask the model.

In [2]:
cache_file = CACHE / f"checker_pyes_{PROMPT_VERSION}.csv"
if cache_file.exists() and len(pd.read_csv(cache_file)) == len(ROWS):
    ROWS["p_yes"] = pd.read_csv(cache_file)["p_yes"].values
    SECONDS = float(open(CACHE / f"checker_seconds_{PROMPT_VERSION}.txt").read())
else:
    checker = Checker()
    partial, progress = CACHE / f"checker_pyes_{PROMPT_VERSION}.partial.npy", CACHE / "checker_progress.txt"
    done = list(np.load(partial)) if partial.exists() else []
    t0 = time.time()
    for s in range(len(done), len(ROWS), 500):
        done += checker.p_yes(ROWS["prompt"].iloc[s:s + 500].tolist())
        np.save(partial, np.array(done))
        progress.write_text(f"{len(done)} / {len(ROWS)} questions, {time.time() - t0:.0f} s")
    SECONDS = time.time() - t0
    ROWS["p_yes"] = done
    ROWS[["record", "unit_index", "label", "p_yes"]].to_csv(cache_file, index=False)
    open(CACHE / f"checker_seconds_{PROMPT_VERSION}.txt", "w").write(str(SECONDS))
    partial.unlink()
print(f"{len(ROWS):,} questions in {SECONDS:.0f} s on the RTX 4070 ({len(ROWS) / SECONDS:.1f} questions per second)")
print(f"share answered yes (≥ 0.5): {(ROWS['p_yes'] >= 0.5).mean():.1%}")

C:\Work\claude\watchlistRiskClassification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1012 18:18:48.008000 10300 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 398/398 [00:06<00:00, 64.72it/s]


3,430 questions in 457 s on the RTX 4070 (7.5 questions per second)
share answered yes (≥ 0.5): 55.2%


**Victim check (from v3).** One extra question per record that has flags: is the listed person a victim or missing
person rather than a suspect? If yes (≥ 0.5), the record gets no text-layer labels.

In [3]:
FLAGGED = sorted(ROWS["record"].unique())
role_file = CACHE / f"checker_role_{PROMPT_VERSION}.csv"
if PROMPT_VERSION < "v3":
    P_VICTIM = pd.Series(0.0, index=FLAGGED)
elif role_file.exists():
    P_VICTIM = pd.read_csv(role_file, index_col="record")["p_victim"]
else:
    try:
        checker
    except NameError:
        checker = Checker()
    P_VICTIM = pd.Series(checker.p_yes(role_questions(df["text_noname"].iloc[FLAGGED].tolist())), index=FLAGGED, name="p_victim")
    P_VICTIM.rename_axis("record").to_csv(role_file)
VICTIMS = P_VICTIM.index[P_VICTIM >= 0.5]
print(f"records judged victim / missing person: {len(VICTIMS)} of {len(FLAGGED)}")
df.loc[VICTIMS, ["source", "text_noname"]].assign(text_noname=lambda x: x["text_noname"].str[:160])

records judged victim / missing person: 13 of 642


,source,text_noname
162,EU-TRAVEL-BAN,Entity type: Individual\nPrograms: Sanctions ; Council Decision concerning restrictive measures against ISIL (Da'esh...
171,FBI-WANTED,Entity type: Individual\nPrograms: Law Enforcement ; Seeking Information ; US\nAdditional info: Reward ; The FBI is ...
174,FBI-WANTED,Entity type: Individual\nPrograms: Law Enforcement ; Kidnappings and Missing Persons ; US\nRemarks: Summary ; Januar...
180,FBI-WANTED,Entity type: Individual\nPrograms: Law Enforcement ; ViCAP Homicides and Sexual Assaults ; US\nRemarks: Summary ; De...
434,FBI-WANTED,Entity type: Individual\nPrograms: Law Enforcement ; ViCAP Homicides and Sexual Assaults ; US\nRemarks: Summary ; UN...
435,FBI-WANTED,Entity type: Individual\nPrograms: Law Enforcement ; Seeking Information ; US\nAdditional info: Reward ; The FBI is ...
552,FBI-WANTED,Entity type: Individual\nPrograms: Law Enforcement ; Seeking Information ; US\nAdditional info: Reward ; The FBI is ...
553,FBI-WANTED,Entity type: Individual\nPrograms: Law Enforcement ; Seeking Information ; US\nAdditional info: Reward ; The FBI is ...
554,FBI-WANTED,Entity type: Individual\nPrograms: Law Enforcement ; Kidnappings and Missing Persons ; US || Law Enforcement ; India...
557,FBI-WANTED,"Entity type: Individual\nPrograms: Law Enforcement ; ViCAP Missing Persons ; US\nRemarks: Summary ; April 26, 2010 |..."


## 2. From unit answers to record labels
Record score for a label = the highest P(yes) among its flagged units (0 if no unit was flagged).

In [4]:
S = np.zeros(Y.shape)
for r, label, p in ROWS[["record", "label", "p_yes"]].itertuples(index=False):
    j = d.labels.index(label)
    S[r, j] = max(S[r, j], p)
S[VICTIMS] = 0

FOLDS = d.unseen_folds()
P_fixed = (S >= 0.5).astype(int)
P_tuned = np.zeros_like(Y)
P_tuned[:, SCOPE] = crossfit_predict(S[:, SCOPE], Y[:, SCOPE], FOLDS)
P_flags = (S > 0).astype(int)

nb02 = np.load(CACHE / "nb02_predictions.npz")
APPROACHES = {
    "TF-IDF + LR (notebook 01/02), unseen lists": nb02["B0 char TF-IDF + LR | unseen"],
    "best whole-record model: TF-IDF + e5-base (notebook 02), unseen lists": nb02["ENS B0 + E2 mE5-base | unseen"],
    "highlighter only (every flag = yes)": P_flags,
    "highlighter + checker, cut-off 0.5 (no tuning)": P_fixed,
    "highlighter + checker, cut-off per label tuned on other folds": P_tuned,
}

def view(P):
    pr, rc, f1 = ml_part(Y, P, L, SCOPE)
    sys_ = P | L
    return {"ML part P": pr, "ML part R": rc, "ML part F1": f1,
            "macro-F1, 15 labels (lookup ∪ model)": np.mean([prf(Y[:, i], sys_[:, i])[2] for i in SCOPE]),
            "labels added": int((P[:, SCOPE] & (1 - L[:, SCOPE])).sum()),
            "added but wrong": int((P[:, SCOPE] & (1 - L[:, SCOPE]) & (1 - Y[:, SCOPE])).sum())}

card = pd.DataFrame({k: view(P) for k, P in APPROACHES.items()}).T
card.to_csv(RESEARCH / "results" / "checker_score_card.csv")
card

,ML part P,ML part R,ML part F1,"macro-F1, 15 labels (lookup ∪ model)",labels added,added but wrong
"TF-IDF + LR (notebook 01/02), unseen lists",0.483,0.674,0.563,0.645,851.0,440.0
"best whole-record model: TF-IDF + e5-base (notebook 02), unseen lists",0.518,0.697,0.594,0.680,820.0,395.0
highlighter only (every flag = yes),0.502,0.941,0.655,0.758,1143.0,569.0
"highlighter + checker, cut-off 0.5 (no tuning)",0.763,0.834,0.797,0.859,667.0,158.0
"highlighter + checker, cut-off per label tuned on other folds",0.739,0.833,0.783,0.844,687.0,179.0


**Reading — score card (15 labels, label instances the lookup misses; original reference)**

* Question **v2** (`PROMPT_VERSION`): adds the taxonomy's sample indicators, "victims, officials fighting the conduct
  and list authorities do not count", "one event can carry several labels", and a PEP note on non-state armed groups.
  v1 → v2: precision 0.706 → **0.733**, recall 0.867 → **0.875**, wrong labels added 220 → 195.
* **Highlighter + checker beats every whole-record model:** ML-part F1 **0.798** vs 0.594 for the best notebook 02 model
  on unseen lists (precision 0.733 vs 0.518, recall 0.875 vs 0.697).
* The checker removes 69% of the highlighter's wrong flags (636 → 195) and keeps 92% of its correct ones.
* A per-label tuned cut-off is no better (F1 0.795) than the fixed 0.5 — no labelled data is needed to set it.
* v2 was written after looking at v1's errors on these same records, so its gain here is slightly optimistic.

## 2b. Scores against the reviewed reference
Every case where the checker and the reference disagree was reviewed against the taxonomy definitions
(`data/reference_review.csv`: decision `keep` / `add` / `remove` with a reason per case). A random sample of 40 cases
where both said yes was also reviewed: all 40 were correct, so cases where both are wrong are rare.
Only the reference changes here; the checker's answers are the same.

In [5]:
review = pd.read_csv(RESEARCH / "data" / "reference_review.csv", encoding="utf-8-sig")
print(review["decision"].value_counts().to_dict())
key = {k: i for i, k in enumerate(df["record_key"])}
YR = Y.copy()
for k, label, v in review[["record_key", "label", "reviewed"]].itertuples(index=False):
    YR[key[k], d.labels.index(label)] = v

def reviewed_view(P):
    pr, rc, f1 = ml_part(YR, P, L, SCOPE)
    w, s = (df["segment"] == "Wanted lists").values, (df["segment"] == "Sanctions & terror lists").values
    return {"ML part P": pr, "ML part R": rc, "ML part F1": f1,
            "wanted lists P": ml_part(YR[w], P[w], L[w], SCOPE)[0], "sanctions lists P": ml_part(YR[s], P[s], L[s], SCOPE)[0]}

reviewed_card = pd.DataFrame({k: reviewed_view(APPROACHES[k]) for k in (
    "best whole-record model: TF-IDF + e5-base (notebook 02), unseen lists", "highlighter + checker, cut-off 0.5 (no tuning)")}).T
reviewed_card.to_csv(RESEARCH / "results" / "checker_score_card_reviewed.csv")
reviewed_card

{'keep': 176, 'add': 60, 'remove': 3}


,ML part P,ML part R,ML part F1,wanted lists P,sanctions lists P
"best whole-record model: TF-IDF + e5-base (notebook 02), unseen lists",0.540,0.664,0.596,0.607,0.463
"highlighter + checker, cut-off 0.5 (no tuning)",0.843,0.843,0.843,0.943,0.754


In [6]:
rows = []
for i in SCOPE:
    added = P_fixed[:, i] & (1 - L[:, i])
    good, n = int((added & YR[:, i]).sum()), int(added.sum())
    need = int((YR[:, i] & (1 - L[:, i])).sum())
    lo, _ = wilson(good, n)
    rows.append({"label": d.short[i], "labels added": n, "precision": good / max(n, 1), "precision, 95% lower bound": lo,
                 "recall": int((added & YR[:, i]).sum()) / max(need, 1), "needed": need})
reviewed_per_label = pd.DataFrame(rows).set_index("label").sort_values("precision", ascending=False)
reviewed_per_label.to_csv(RESEARCH / "results" / "checker_per_label_reviewed.csv")
reviewed_per_label

,labels added,precision,"precision, 95% lower bound",recall,needed
label,,,,,
Fraud,41,1.000,0.914,0.745,55
Money Laundering,55,0.945,0.851,0.981,53
Violent Crime,104,0.942,0.880,0.867,113
Corruption,16,0.938,0.717,0.833,18
Theft,43,0.930,0.814,0.741,54
Proliferation,21,0.905,0.711,0.905,21
Cybercrime,9,0.889,0.565,0.889,9
Judicial,92,0.870,0.786,0.816,98
Tax Crime,22,0.818,0.615,1.000,18


**Reading — reviewed reference**

* Review of the 239 disagreements: the reference was right in **176** (checker wrong), missed the label in **60**, and had
  a wrong label in **3** (Proliferation for explosive attacks called "weapons of mass destruction" in US law).
  Typical reference gaps: Iranian nuclear-programme designations → Proliferation; court charges / probation /
  hearings → Judicial; killing and attacks on civilians → Violent Crime (alongside Human Rights).
* **Against the reviewed reference: precision 0.815, recall 0.891, F1 0.851** (best whole-record model: 0.540 / 0.664).
  Wanted lists precision **0.920**; sanctions lists **0.714**.
* Per label: Fraud 1.00, Corruption 0.94, Theft 0.94, Money Laundering 0.93 are at or above 90%; Cybercrime 0.89,
  Judicial 0.87, Violent Crime 0.86 are close; **PEP 0.59** and **Terrorism 0.52** are far below.
* The checker's remaining mistakes on those two are systematic: election **candidates** and the **listing authority**
  ("Australian Government Department of Foreign Affairs") called PEP; **Taliban** roles without any reported terrorist
  act called Terrorism; and **victims** ("Homicide Victim", "Missing Persons") called Violent Crime.
* Caveat: the review was done by Claude against the taxonomy, not by a human; reasons are in the file for spot-checking.

## 3. Per label (cut-off 0.5)

In [7]:
rows = []
for i in SCOPE:
    need = NEED[:, i]
    out = {}
    for name, P in (("best whole-record", APPROACHES["best whole-record model: TF-IDF + e5-base (notebook 02), unseen lists"]),
                    ("checker", P_fixed)):
        pr, rc, f1 = ml_part(Y, P, L, [i])
        out[(name, "P")], out[(name, "R")], out[(name, "F1")] = pr, rc, f1
    found = int((need & P_fixed[:, i]).sum())
    lo, hi = wilson(found, int(need.sum()))
    out[("checker", "R 95% CI")] = f"{lo:.2f}–{hi:.2f}"
    out[("", "needed")], out[("", "tier")] = int(need.sum()), TIERS[i]
    rows.append(pd.Series(out, name=d.short[i]))
per_label = pd.DataFrame(rows)[[("", "tier"), ("", "needed"), ("best whole-record", "P"), ("best whole-record", "R"),
                                ("best whole-record", "F1"), ("checker", "P"), ("checker", "R"), ("checker", "F1"),
                                ("checker", "R 95% CI")]]
per_label.to_csv(RESEARCH / "results" / "checker_per_label.csv")
per_label.sort_values(("", "needed"), ascending=False)

best whole-record               checker                         
                              tier needed                 P      R     F1       P      R     F1   R 95% CI
Violent Crime       A validated ML     95             0.608  0.768  0.679   0.788  0.863  0.824  0.78–0.92
Judicial            A validated ML     89             0.705  0.753  0.728   0.783  0.809  0.796  0.72–0.88
PEP                 A validated ML     67             0.303  0.910  0.455   0.548  0.687  0.609  0.57–0.78
Fraud               A validated ML     54             0.860  0.796  0.827   0.976  0.741  0.842  0.61–0.84
Money Laundering    A validated ML     53             0.825  0.623  0.710   0.945  0.981  0.963  0.90–1.00
Theft               A validated ML     53             0.343  0.434  0.383   0.930  0.755  0.833  0.62–0.85
Organized Crime   B provisional ML     37             0.538  0.568  0.553   0.750  0.811  0.779  0.66–0.91
Trafficking       B provisional ML     36             0.667  0.833  0.741   0.795  0.972  0.875  0.86–1.00
Forgery           B provisional ML     34             0.512  0.618  0.560   0.721  0.912  0.805  0.77–0.97
Human Rights               C no ML     23             0.432  0.696  0.533   0.667  0.957  0.786  0.79–0.99
Corruption                 C no ML     17             0.381  0.471  0.421   0.875  0.824  0.848  0.59–0.94
Proliferation              C no ML     17             0.458  0.647  0.537   0.619  0.765  0.684  0.53–0.90
Tax Crime                  C no ML     17             0.500  0.412  0.452   0.773  1.000  0.872  0.82–1.00
Terrorism                  C no ML     12             0.750  0.500  0.600   0.500  0.833  0.625  0.55–0.95
Cybercrime                 C no ML      6             0.278  0.833  0.417   0.556  0.833  0.667  0.44–0.97

**Reading — per label (original reference)**

* Large gains over the best whole-record model: Theft F1 0.38 → **0.85**, Money Laundering 0.71 → **0.95**, Fraud
  0.83 → **0.90**, Judicial 0.73 → **0.88**, Corruption 0.42 → **0.85**, Tax 0.45 → **0.85**, Trafficking 0.74 → **0.88**.
* Weak: **Terrorism** (precision 0.36), **Proliferation** (0.46), **PEP** (0.53).

## 4. Per segment and list (cut-off 0.5)

In [8]:
seg = []
for key in ("segment", "source"):
    for g, idx in df.groupby(key).groups.items():
        idx = np.asarray(idx)
        if NEED[np.ix_(idx, SCOPE)].sum() < 10:
            continue
        b = ml_part(Y[idx], APPROACHES["best whole-record model: TF-IDF + e5-base (notebook 02), unseen lists"][idx], L[idx], SCOPE)
        c = ml_part(Y[idx], P_fixed[idx], L[idx], SCOPE)
        seg.append({"level": key, "group": g, "needed": int(NEED[np.ix_(idx, SCOPE)].sum()),
                    "best whole-record F1": b[2], "checker P": c[0], "checker R": c[1], "checker F1": c[2]})
seg = pd.DataFrame(seg).set_index(["level", "group"])
seg.sort_values("needed", ascending=False)

needed  best whole-record F1  checker P  checker R  checker F1
level   group                                                                                               
segment Wanted lists                             377                 0.656      0.909      0.844       0.875
        Sanctions & terror lists                 219                 0.514      0.614      0.822       0.703
source  INTERPOL-RED-NOTICES                     114                 0.630      0.948      0.807       0.872
        FBI-WANTED                               108                 0.581      0.901      0.759       0.824
        UKSL                                      65                 0.516      0.746      0.815       0.779
        US-MARSHALS-PROFILED-FUGITIVES            63                 0.722      0.831      0.937       0.881
        EU-MOST-WANTED                            59                 0.774      0.900      0.915       0.908
        SECO-SANCTIONS                            50                 0.503      0.537      0.880       0.667
        NCA-MOST-WANTED                           33                 0.703      1.000      0.939       0.969
        DFAT                                      22                 0.565      0.559      0.864       0.679
        UN-SANCTIONS                              19                 0.667      0.500      0.842       0.627
        EU-FINANCIAL-SANCTIONS                    19                 0.750      0.538      0.737       0.622
        EU-TRAVEL-BAN                             16                 0.491      0.619      0.812       0.703
segment Regulatory / debarment                    14                 0.727      1.000      0.786       0.880
source  ATC-DESIGNATED-TERRORIST-INDIVIDUALS      12                 0.556      1.000      0.833       0.909
        OFAC-SDN                                  12                 0.128      1.000      0.833       0.909

**Reading — segments and lists (original reference)**

* **Wanted lists: F1 0.893** (precision 0.881) vs 0.656 for the best whole-record model.
* **Sanctions & terror lists: F1 0.678** (precision 0.572) vs 0.514 — the weak point (DFAT 0.46, UN 0.44, EU financial
  0.45, SECO 0.50). Section 2b shows how much of this is reference gaps.

## 5. What the checker gets right and wrong — real examples
For a few cases the model is also asked for a one-sentence reason (text generation, slower), to show the evidence a
reviewer would see.

In [9]:
ROWS["truth"] = [Y[r, d.labels.index(l)] for r, l in zip(ROWS["record"], ROWS["label"])]
ROWS["from lookup"] = [L[r, d.labels.index(l)] for r, l in zip(ROWS["record"], ROWS["label"])]
judge = ROWS[ROWS["from lookup"] == 0]
best_unit = judge.loc[judge.groupby(["record", "label"])["p_yes"].idxmax()]
cases = {
    "kept, correct": best_unit[(best_unit["p_yes"] >= 0.5) & (best_unit["truth"] == 1)],
    "dropped, correct": best_unit[(best_unit["p_yes"] < 0.5) & (best_unit["truth"] == 0)],
    "kept, but the label is not in the reference": best_unit[(best_unit["p_yes"] >= 0.5) & (best_unit["truth"] == 0)],
    "dropped, but the label is in the reference": best_unit[(best_unit["p_yes"] < 0.5) & (best_unit["truth"] == 1)],
}
print({k: len(v) for k, v in cases.items()})
examples = pd.concat({k: v.sample(min(4, len(v)), random_state=1) for k, v in cases.items()})
examples["label"] = examples["label"].map(lambda l: d.short[d.labels.index(l)])
examples[["label", "field", "unit", "p_yes"]]

{'kept, correct': 511, 'dropped, correct': 422, 'kept, but the label is not in the reference': 168, 'dropped, but the label is in the reference': 65}


label            field                                                                                                                     unit      p_yes
kept, correct                               310       Trafficking          Remarks                                                                                                 Federal Drug Trafficking  1.000e+00
                                            2202            Theft          Remarks  Under Christ-Roi Chéry’s command, the Ti Bwa gang is engaged in violence and criminal activities in Haiti, including...  1.000e+00
                                            1836          Forgery         Programs                                                                                                          Invoice forgery  1.000e+00
                                            1505            Theft          Remarks                                                                                     Robbery in Band with Triple Homicide  1.000e+00
dropped, correct                            3159              PEP    Relationships                                                                                                    Leader or official of  1.152e-02
                                            1341  Organized Crime         Programs                                                                        Gang-type fraud on a repetitive and gainful basis  5.702e-02
                                            249         Terrorism      Occupations                                                                           Chief of Kabul police under the Taliban regime  7.340e-07
                                            1432          Forgery          Remarks  Marina Gartung is the majority shareholder of Public Joint Stock Company “Chelyabinsk Forge and Press Plant” (PJSC C...  9.094e-09
kept, but the label is not in the reference 759               PEP      Source note                                                                                                        The TPD President  1.000e+00
                                            2069              PEP          Remarks  The Foreign Minister of Russia, Sergey Lavrov has confirmed the Foundation’s consistent work in countering xenophobi...  1.000e+00
                                            95                PEP      Occupations                                                                                                   SENATOR of PHILIPPINES  1.000e+00
                                            2763              PEP      Source note  Continued in effect for a further three years by Declaration of the Minister for Foreign Affairs made on 23 May 2020...  9.999e-01
dropped, but the label is in the reference  2908            Fraud          Remarks  Jin Sung-Il and Pak Jin-Song are wanted for their alleged involvement in a fraudulent scheme to obtain remote inform...  1.759e-02
                                            3112         Judicial  Additional info  The FBI is offering a reward of up to $100,000 for information leading directly to the arrest and conviction of Eliz...  3.928e-04
                                            2582            Theft         Programs  Illicit trafficking in narcotic drugs and psychotropic substances, Organised or armed robbery, Participation in a cr...  7.236e-02
                                            318          Judicial          Remarks  On July 17, 2025, a federal arrest warrant was issued for this individual in the United States District Court, Distr...  1.776e-01

In [10]:
try:
    checker
except NameError:
    checker = Checker()
reasons = examples.groupby(level=0, group_keys=False).head(2)
for (case, idx), row in reasons.iterrows():
    print(f"[{case}] {row['label']} | P(yes)={row['p_yes']:.2f} | {row['unit'][:150]}")
    print("   model:", checker.explain(ROWS.loc[idx, "prompt"]), "\n")

[kept, correct] Trafficking | P(yes)=1.00 | Federal Drug Trafficking
   model: yes  
The text "Federal Drug Trafficking" directly matches the definition of drug/controlled-substance trafficking, which is a core component of the Crime-Trafficking & Smuggling label. 

[kept, correct] Theft | P(yes)=1.00 | Under Christ-Roi Chéry’s command, the Ti Bwa gang is engaged in violence and criminal activities in Haiti, including murder, robbery, rape, hijacking 
   model: Yes  
The text explicitly mentions robbery ("hijacking of goods and trucks") and other property-related criminal activities, which aligns with the definition of Crime-Theft & Property Crime, including unlawful appropriation of property. 

[dropped, correct] PEP | P(yes)=0.01 | Leader or official of
   model: yes

The phrase "Leader or official of" in the context of a "WAKED MONEY LAUNDERING ORGANIZATION" suggests a prominent public function, as leadership or official status implies a senior role, which aligns with the PEP defini

**Reading — examples**

* Against the original reference: 534 kept correctly, 441 dropped correctly, 195 kept but not in the reference, 44
  dropped although in the reference. Section 2b reviews every one of these 239 disagreements.
* **The model's written reasons are still not trustworthy** for audit: the decision uses the score, and the evidence
  shown to a reviewer is the flagged sentence itself.

## 6. Speed and cost

In [11]:
per_q = SECONDS / len(ROWS)
cost = pd.Series({
    "questions per 1,000 records": len(ROWS) / len(df) * 1000,
    "seconds per question (RTX 4070, 4-bit, batches of 4)": per_q,
    "minutes per 1,000 records": len(ROWS) / len(df) * 1000 * per_q / 60,
    "hours for 300k records, no cache": 300 * len(ROWS) / len(df) * 1000 * per_q / 3600,
    "hours for 300k records, cache of identical (unit, label) pairs": 300 * ROWS[["unit", "label"]].drop_duplicates().shape[0] / len(df) * 1000 * per_q / 3600,
})
cost.round(2)

questions per 1,000 records                                       3811.11
seconds per question (RTX 4070, 4-bit, batches of 4)                 0.13
minutes per 1,000 records                                            8.46
hours for 300k records, no cache                                    42.32
hours for 300k records, cache of identical (unit, label) pairs      25.11
dtype: float64

**Reading — speed and cost**

* v2 questions are longer (indicators added): **0.15 s per question** on the RTX 4070 → ≈ **9.4 minutes per 1,000 records**;
  300k records ≈ 47 h without a cache, ≈ 27 h reusing answers for identical (unit, label) pairs.
* On the laptop **CPU** the same model takes ≈ 24 s per question (measured on 16 questions) — ≈ 25 hours per 1,000
  records, so a CPU-only server needs a smaller model (see conclusions).

## 7. Conclusions

1. **Highlighter + checker is the right design:** against the reviewed reference, precision **0.82**, recall **0.89**,
   F1 **0.85**, vs 0.54 / 0.66 / 0.60 for the best whole-record model — with no training on our labels.
2. **Wanted lists already meet a 90% precision target (0.92); sanctions lists do not (0.71).**
3. 4 labels are at ≥ 90% precision (Fraud, Corruption, Theft, Money Laundering); PEP (0.59) and Terrorism (0.52) pull
   the total down with systematic, fixable mistakes (candidates, list authorities, Taliban roles, victims).
4. Decisions come from the score; evidence is the flagged sentence (this model's written reasons are unreliable).
5. Cost: ≈ 9 min per 1,000 records on the laptop GPU; unusable on CPU (≈ 25 h per 1,000) → a CPU server needs a small
   distilled model.
6. Next for the 90% target: PEP / Terrorism question fixes, a larger checker model, and a check on fresh records
   (fixes made after seeing these errors make scores on these 900 records optimistic).